<a href="https://colab.research.google.com/github/ShreyaNimje/GenAI-Concepts/blob/main/GenAIP4_encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
#define parameters for our dataset
num_samples = 1000 # Number of training examples
input_seq_len = 10 # Length of input sequences
target_seq_len = 10 # Length of target sequence
num_features = 5 # Number of unique 'words' or 'tokens' in our vocabulary

# Generate random input sequences (one-hot encoded for simplicity)
# Each input sequnce will be a series of integer indices
encoder_input_data = np.random.randint(0, num_features, size=(num_samples, input_seq_len))

#Generate target sequences (reversed version of input sequences, with a start-of-sequence token)
# For simplicity we would just reverse them. In real scenario, these would be different.
decoder_target_data = np.zeros((num_samples, target_seq_len, num_features), dtype='float32')
decoder_input_data = np.zeros((num_samples, target_seq_len, num_features), dtype='float32')

# Create one-hot encoded versions for both input and target for keras
encoder_input_data_one_hot = np.zeros((num_samples, input_seq_len, num_features), dtype='float32')
for i, sequence in enumerate(encoder_input_data):
    for t, token_index in enumerate(sequence):
        encoder_input_data_one_hot[i, t, token_index] = 1.0

# Generate target sequence which are reversed inputs, and prepare decoder input/target
for i in range(num_samples):
  reversed_sequence = encoder_input_data[i][::-1]

  decoder_input_data[i, 0 , 0] = 1.0
  for t, feature in enumerate(reversed_sequence[:-1]):
    decoder_input_data[i, t + 1, feature] = 1.0

  for t, feature in enumerate(reversed_sequence):
    decoder_target_data[i, t, feature] = 1.0

print("Encoder Input Shape :", encoder_input_data_one_hot.shape)
print("Decoder Input Shape :", decoder_input_data.shape)
print("Decoder Target Shape :", decoder_target_data.shape)

print("\nSample Encoder Input (indices):")
print(encoder_input_data[0])
print("Sample Decoder Target (one-hot, indicating indices):")
print(np.argmax(decoder_target_data[0], axis=-1))

Encoder Input Shape : (1000, 10, 5)
Decoder Input Shape : (1000, 10, 5)
Decoder Target Shape : (1000, 10, 5)

Sample Encoder Input (indices):
[4 3 4 1 2 1 2 0 4 4]
Sample Decoder Target (one-hot, indicating indices):
[4 4 0 2 1 2 1 4 3 4]


In [ ]:
# Encoder Setup
latent_dim = 256
encoder_inputs = Input(shape=(None, num_features))
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

In [ ]:
# Decoder Setup
decoder_inputs = Input(shape=(None, num_features))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_features, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 5)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 5)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    268,288 │ input_layer[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    268,288 │ input_layer_1[0]… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 5)   │      1,285 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 537,861 (2.05 MB)

 Trainable params: 537,861 (2.05 MB)

 Non-trainable params: 0 (0.00 B)